# Numin2 Stock Ranking Meta-Learning Demo
**Interactive evaluation of trained meta-learning models for stock rank prediction**

This notebook lets you:
1. Choose a trained model (ANIL, Attention MAML, ProtoNet, CNP, FOMAML, Reptile, etc.)
2. Pick a test month from the dataset **or provide custom stock data**
3. Watch the model adapt from 5 support days and predict stock rankings
4. Visualize predicted vs actual rankings with Spearman correlation

## Environment Setup
Use the **`base`** conda environment:
```bash
conda activate base
jupyter notebook
```
Required packages (all pre-installed in base): `torch 2.7+`, `scipy`, `pandas`, `numpy`, `matplotlib`, `tqdm`, `ipywidgets`

In [3]:
import torch
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
import json, os, sys, random
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

sys.path.insert(0, '.')
random.seed(42); np.random.seed(42); torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


## 1. Load Dataset

In [4]:
from numin_reptile import NuminDataset

dataset = NuminDataset('numin_sample.parquet')

# Chronological split
n = len(dataset)
train_end = int(0.7 * n)
val_end = int(0.85 * n)
test_idx = list(range(val_end, n))

print(f'Total tasks (months): {n}')
print(f'Test tasks: {len(test_idx)}')
print(f'Each task: support={dataset.support_days} days, window={dataset.window_size}')

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.

## 2. Available Models

In [ ]:
MODEL_CONFIGS = {
    'ANIL (corr=0.670)': {
        'module': 'numin_anil', 'dataset_cls': 'NuminDataset',
        'model_cls': 'ANILModel', 'model_args': {'num_stocks': 50, 'hidden_dim': 256},
        'ckpt': 'checkpoints_numin_anil/best_model.pt',
        'inner_steps': 3, 'inner_lr': 0.01, 'head_only': True,
    },
    'FOMAML (corr=0.631)': {
        'module': 'numin_fomaml', 'dataset_cls': 'NuminDataset',
        'model_cls': 'Model', 'model_args': {'ns': 50, 'hd': 256},
        'ckpt': 'checkpoints_numin_fomaml/best_model.pt',
        'inner_steps': 5, 'inner_lr': 0.01, 'head_only': False,
    },
    'Reptile (corr=0.614)': {
        'module': 'numin_reptile', 'dataset_cls': 'NuminDataset',
        'model_cls': 'NuminModel', 'model_args': {'num_stocks': 50, 'hidden_dim': 256},
        'ckpt': 'checkpoints_numin_reptile/best_model.pt',
        'inner_steps': 10, 'inner_lr': 0.01, 'head_only': False,
    },
    'CNP (corr=0.634)': {
        'module': 'numin_cnp', 'dataset_cls': 'NuminDataset',
        'model_cls': 'CNPModel', 'model_args': {'ns': 50, 'hd': 256},
        'ckpt': 'checkpoints_numin_cnp/best_model.pt',
        'inner_steps': 0, 'inner_lr': 0, 'head_only': False,
    },
    'ProtoNet (corr=0.639)': {
        'module': 'numin_protonet', 'dataset_cls': 'NuminDataset',
        'model_cls': 'NuminProtoNet', 'model_args': {'num_stocks': 50, 'hidden_dim': 256},
        'ckpt': 'checkpoints_numin_protonet/best_model.pt',
        'inner_steps': 0, 'inner_lr': 0, 'head_only': False,
    },
}

def load_model(config):
    mod = __import__(config['module'])
    cls = getattr(mod, config['model_cls'])
    model = cls(**config['model_args'])
    model.load_state_dict(torch.load(config['ckpt'], map_location=device, weights_only=True))
    return model.to(device)

print('Available models:')
for name in MODEL_CONFIGS:
    print(f'  - {name}')

## 3. Choose Model

In [ ]:
# ============================
# CHOOSE YOUR MODEL HERE
# ============================
MODEL_NAME = 'ANIL (corr=0.670)'  # <-- Change this

config = MODEL_CONFIGS[MODEL_NAME]
print(f'Loading {MODEL_NAME}...')
model = load_model(config)
print(f'Loaded! Parameters: {sum(p.numel() for p in model.parameters()):,}')

## 4. Inference & Visualization

In [ ]:
def adapt_and_predict(model, task, config):
    """Adapt model on support set, predict on query set."""
    ss = task['support_samples'].to(device)
    st = task['support_targets'].to(device)
    qs = task['query_samples'].to(device)
    qt = task['query_targets'].to(device)
    ns = 50
    
    inner_steps = config['inner_steps']
    
    if inner_steps > 0:
        # Save weights
        ow = {n: p.data.clone() for n, p in model.named_parameters()}
        if config.get('head_only'):
            params = [p for n, p in model.named_parameters() if 'head' in n or 'dec' in n or 'decoder' in n]
        else:
            params = list(model.parameters())
        iopt = optim.SGD(params, lr=config['inner_lr'])
        model.train()
        for _ in range(inner_steps):
            logits = model(ss)
            loss = F.cross_entropy(logits.reshape(-1, ns), st.reshape(-1))
            iopt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 5.0)
            iopt.step()
        
        model.eval()
        with torch.no_grad():
            q_logits = model(qs)
        
        # Restore
        with torch.no_grad():
            for n, p in model.named_parameters(): p.data = ow[n]
    else:
        # No inner loop (CNP, ProtoNet)
        model.eval()
        with torch.no_grad():
            q_logits = model(ss, st, qs) if hasattr(model, 'encode_support') else model(qs)
    
    # Soft expected rank for Spearman
    rank_indices = torch.arange(ns, device=device).float()
    pred_ranks = (q_logits.softmax(-1) * rank_indices).sum(-1)
    
    return pred_ranks.cpu().numpy(), qt.cpu().numpy()

def visualize_ranking(pred_ranks, true_ranks, day_idx=0, top_k=15):
    """Visualize predicted vs actual stock rankings for one day."""
    pred = pred_ranks[day_idx]
    true = true_ranks[day_idx]
    
    corr, pval = spearmanr(pred, true)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'Stock Ranking Prediction (Day {day_idx+1}) | Spearman r = {corr:.4f} (p={pval:.4f})',
                 fontsize=13, fontweight='bold')
    
    # Plot 1: Scatter plot
    ax = axes[0]
    ax.scatter(true, pred, alpha=0.6, c='steelblue', edgecolors='navy', s=40)
    ax.plot([0, 49], [0, 49], 'r--', alpha=0.5, label='Perfect prediction')
    ax.set_xlabel('True Rank'); ax.set_ylabel('Predicted Rank')
    ax.set_title('Predicted vs True Rank'); ax.legend(); ax.grid(True, alpha=0.3)
    
    # Plot 2: Top/Bottom stocks comparison
    ax = axes[1]
    true_order = np.argsort(true)
    pred_order = np.argsort(pred)
    show_idx = list(true_order[:top_k])  # top stocks by true rank
    x = np.arange(len(show_idx))
    ax.barh(x - 0.2, true[show_idx], 0.35, label='True Rank', color='steelblue', alpha=0.8)
    ax.barh(x + 0.2, pred[show_idx], 0.35, label='Predicted Rank', color='coral', alpha=0.8)
    ax.set_yticks(x); ax.set_yticklabels([f'Stock {i}' for i in show_idx], fontsize=8)
    ax.set_xlabel('Rank (0=best)'); ax.set_title(f'Top {top_k} Stocks by True Rank')
    ax.legend(); ax.invert_yaxis()
    
    # Plot 3: Rank difference histogram
    ax = axes[2]
    diffs = pred - true
    ax.hist(diffs, bins=20, color='steelblue', alpha=0.7, edgecolor='navy')
    ax.axvline(0, color='red', linestyle='--', alpha=0.7)
    ax.set_xlabel('Rank Difference (Predicted - True)')
    ax.set_ylabel('Count'); ax.set_title(f'Prediction Error Distribution\nMAE={np.mean(np.abs(diffs)):.1f}')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    return corr

print('Visualization ready.')

## 5A. Test on Dataset Months

In [ ]:
# ============================
# CHOOSE TEST MONTH
# ============================
MONTH_IDX = 'random'  # <-- Set to 0, 1, 2, 3 or 'random'

if MONTH_IDX == 'random':
    MONTH_IDX = random.randint(0, len(test_idx) - 1)

task = dataset[test_idx[MONTH_IDX]]
print(f'Test month {MONTH_IDX}: support={task["support_samples"].shape[0]} days, query={task["query_samples"].shape[0]} days')

pred_ranks, true_ranks = adapt_and_predict(model, task, config)

# Show results for first 3 query days
correlations = []
for day in range(min(3, len(pred_ranks))):
    corr = visualize_ranking(pred_ranks, true_ranks, day_idx=day)
    correlations.append(corr)

# Overall correlation for this month
all_corrs = []
for day in range(len(pred_ranks)):
    c, _ = spearmanr(pred_ranks[day], true_ranks[day])
    if not np.isnan(c): all_corrs.append(c)

print(f'\n=== Month Summary ===')
print(f'Average Spearman correlation: {np.mean(all_corrs):.4f} (+/- {np.std(all_corrs):.4f})')
print(f'Best day:  {max(all_corrs):.4f}')
print(f'Worst day: {min(all_corrs):.4f}')

In [ ]:
# Evaluate on ALL test months
print(f'Evaluating {MODEL_NAME} on all {len(test_idx)} test months...\n')

month_corrs = []
for i, idx in enumerate(test_idx):
    task = dataset[idx]
    pred_ranks, true_ranks = adapt_and_predict(model, task, config)
    corrs = []
    for d in range(len(pred_ranks)):
        c, _ = spearmanr(pred_ranks[d], true_ranks[d])
        if not np.isnan(c): corrs.append(c)
    mc = np.mean(corrs) if corrs else 0
    month_corrs.append(mc)
    print(f'  Month {i}: corr = {mc:.4f}')

print(f'\n=== Overall: {np.mean(month_corrs):.4f} (+/- {np.std(month_corrs):.4f}) ===')

## 5B. Custom Stock Data
Provide your own stock returns matrix and see the model rank them.

In [ ]:
# ============================
# CREATE CUSTOM DATA
# ============================
# Shape: (num_days, 50) -- daily returns for 50 stocks
# We need at least window_size(50) + support_days(5) + 1 query day = 56 days

# Option 1: Random synthetic data
np.random.seed(123)
num_days = 60
num_stocks = 50
custom_returns = np.random.randn(num_days, num_stocks) * 0.02  # 2% daily vol

# Add some structure: first 10 stocks trend up, last 10 trend down
for i in range(10):
    custom_returns[:, i] += 0.005  # slight upward bias
for i in range(40, 50):
    custom_returns[:, i] -= 0.005  # slight downward bias

# Create windowed samples
window_size = 50
support_days = 5

samples = []
targets = []
for i in range(window_size, len(custom_returns)):
    window = custom_returns[i - window_size:i]
    next_ret = custom_returns[i]
    ranks = np.argsort(np.argsort(-next_ret))  # higher return = lower rank
    samples.append(window)
    targets.append(ranks)

samples = np.array(samples, dtype=np.float32)
targets = np.array(targets, dtype=np.int64)

# Normalize using support stats
support_data = samples[:support_days]
mean, std = support_data.mean(), support_data.std() + 1e-8
samples = (samples - mean) / std

custom_task = {
    'support_samples': torch.tensor(samples[:support_days]),
    'support_targets': torch.tensor(targets[:support_days]),
    'query_samples': torch.tensor(samples[support_days:]),
    'query_targets': torch.tensor(targets[support_days:]),
}

print(f'Custom data: {num_days} days, {num_stocks} stocks')
print(f'Support: {support_days} days, Query: {len(samples) - support_days} days')

pred_ranks, true_ranks = adapt_and_predict(model, custom_task, config)

for day in range(min(2, len(pred_ranks))):
    visualize_ranking(pred_ranks, true_ranks, day_idx=day)

## 6. Compare All Models

In [ ]:
# Compare all models on a single test month
COMPARE_MONTH = 0
task = dataset[test_idx[COMPARE_MONTH]]

print(f'{"Model":<30} {"Mean Corr":>10} {"Std":>8}')
print('-' * 50)

model_results = {}
for name, cfg in MODEL_CONFIGS.items():
    try:
        m = load_model(cfg)
        pred, true = adapt_and_predict(m, task, cfg)
        corrs = [spearmanr(pred[d], true[d])[0] for d in range(len(pred))]
        corrs = [c for c in corrs if not np.isnan(c)]
        mc = np.mean(corrs)
        model_results[name] = mc
        print(f'{name:<30} {mc:>10.4f} {np.std(corrs):>8.4f}')
        del m
    except Exception as e:
        print(f'{name:<30} ERROR: {e}')

# Bar chart
if model_results:
    fig, ax = plt.subplots(figsize=(10, 4))
    names = list(model_results.keys())
    vals = [model_results[n] for n in names]
    colors = ['#27AE60' if v > 0.6 else '#F39C12' if v > 0.4 else '#E74C3C' for v in vals]
    ax.barh(range(len(names)), vals, color=colors, alpha=0.85, edgecolor='black')
    ax.set_yticks(range(len(names))); ax.set_yticklabels([n.split('(')[0].strip() for n in names])
    ax.set_xlabel('Spearman Correlation'); ax.set_title(f'All Models on Test Month {COMPARE_MONTH}')
    for i, v in enumerate(vals): ax.text(v + 0.01, i, f'{v:.3f}', va='center')
    ax.invert_yaxis(); ax.grid(True, alpha=0.3, axis='x')
    plt.tight_layout(); plt.show()